# LLM - Klasyczne metody klasyfikacji tekstu - LAB

# Zadanie

Zaadaptuj kod z notatnika *LLM - Klasyczne metody klasyfikacji tekstu - Omówienie* do problemu klasyfikacji liczby gwiazdek dla opinii z serwisu Yelp.
Możesz przygotować pętlę treningową albo w czystym PyTorchu, albo z wykorzystaniem biblioteki PyTorch Lightning.

* Wykorzystaj zbiór `Yelp/yelp_review_full` ([link](https://huggingface.co/datasets/Yelp/yelp_review_full)) zawierający opinie z serwisu Yelp (kolumna: `text`) i etykietę (kolumna: `label`) o wartościach $0,1,2,3,4$ określającą liczbę gwiazdek przyznaną przez użytkownika (a ściślej, liczbę gwiazdek minus jeden). Ponieważ mamy pięć klas, ostatnia warstwa liniowa w sieci neuronowej musi zwracać pięć wartości.
    * Zgodnie z dobrą praktyką z części treningowej wydziel dodatkową część walidacyjną i testową.
    * Ogranicz rozmiar każdej części zbioru danych (treningowej, walidacyjnej i testowej). Część treningowa nie powinna zawierać więcej niż 100k elementów.
* Do ekstrakcji cech z tekstu wykorzystaj **metodę TF-IDF** (*term frequency-inverse document frequency*) opartą o podejście typu worek słów (*bag-of-words*). Zastosuj funkcję `TfidfVectorizer` z biblioteki `scikit-learn`.
* Można wykorzystać podobną architekturę sieci (perceptron wielowarstwowy z warstwą Dropout) jak w notatniku wykładowym.



## Punkty do wykonania

1.   Napisz funkcję znajdującą i wyświetlającą $k$ elementów zbioru testowego dla których model najbardziej się myli, czyli estymuje najmniejsze prawdopodobieństwa prawdziwej klasy. Softmax jest funkcją ściśle rosnącą, więc wystarczy znaleźć elementy z najmniejszą wartością nieznormalizowanego wyjścia z sieci (logita) dla prawdziwej klasy.
2.   Zbadaj wpływ wybranych parametrów funkcji ekstrakcji cech z tekstu `TfidfVectorizer` na skuteczność wytrenowanego modelu. Uruchom kilka eksperymentów z różnymi wartościami parametrów i porównaj dokładność wytrenowanego modelu na zbiorze walidacyjnym.
3.   Zbadaj wpływ wybranych hiperparametrów modelu (np. liczba warstw liniowych modelu, rozmiary warstw) i procesu uczenia (np. początkowa wartość stopy uczenia, liczba epok, typ i parametry planisty stopy uczenia, typ i parametry optymalizatora) na skuteczność wytrenowanego modelu. Uruchom kilka eksperymentów z różnymi wartościami hiperparametrów i porównaj dokładność wytrenowanego modelu na zbiorze walidacyjnym. Następnie wykonaj finalną ewaluację najlepszego modelu na zbiorze testowym.


##Przygotowanie środowiska
Upewnij się, że notatnik jest uruchomiony na maszynie z GPU. Jesli GPU nie jest dostępne zmień typ maszyny (Runtime | Change runtime type) i wybierz T4 GPU.

In [2]:
!nvidia-smi

Mon Mar 23 01:17:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.195.03             Driver Version: 570.195.03     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          On  |   00000000:47:00.0 Off |                    0 |
| N/A   23C    P0             51W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

Instalacja dodatkowych bibliotek: datasets (z biblioteki HuggingFace), TorchMetrics i W&B (Weights and Biases) Models.

In [3]:
!pip install -q datasets
!pip install -q torchmetrics
!pip install -q wandb

Import bibliotek.

In [4]:
import torch.nn as nn
import torch
import numpy as np
import matplotlib.pyplot as plt
from datasets import DatasetDict
from sklearn.feature_extraction.text import TfidfVectorizer

print(f"Wersja biblioteki PyTorch: {torch.__version__}")

/net/tscratch/people/plgzwaszczuk/venvs/llm_pw/lib64/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Wersja biblioteki PyTorch: 2.8.0+cu128


Sprawdzenie dostępności GPU.

In [5]:
print(f"Dostępność GPU: {torch.cuda.is_available()}")
print(f"Typ GPU: {torch.cuda.get_device_name(0)}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

Dostępność GPU: True
Typ GPU: NVIDIA A100-SXM4-40GB
cuda


In [16]:
import wandb

# Logowanie do serwisu Weights&Biases monitorującego przebieg eksperymentów
wandb.login()

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /net/people/plgrid/plgzwaszczuk/.netrc
wandb: Currently logged in as: zuza-waszczuk (zuza-waszczuk-knsi-golem) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [17]:
wandb.init(
    project="llm_lab1",
    name="mlp_layer_32",
    config={
        "learning_rate": 0.001,
        "batch_size": 32,
        "epochs": 5,
    }
)

# Rozwiązanie

Podział danych

In [7]:
from datasets import load_dataset
from torch.utils.data import TensorDataset, DataLoader

ds = load_dataset("Yelp/yelp_review_full")

In [8]:
train_val = ds['train'].train_test_split(test_size=0.90, seed=42)
val_test = train_val['test'].train_test_split(test_size=0.98, seed=42)

datasets = DatasetDict({
    'train': train_val['train'],
    'validation': val_test['train'],
    'test': val_test['test'].select(range(11700))
})

In [9]:
print(ds['test'][0])
print(ds['train'][0])

{'label': 0, 'text': 'I got \'new\' tires from them and within two weeks got a flat. I took my car to a local mechanic to see if i could get the hole patched, but they said the reason I had a flat was because the previous patch had blown - WAIT, WHAT? I just got the tire and never needed to have it patched? This was supposed to be a new tire. \\nI took the tire over to Flynn\'s and they told me that someone punctured my tire, then tried to patch it. So there are resentful tire slashers? I find that very unlikely. After arguing with the guy and telling him that his logic was far fetched he said he\'d give me a new tire \\"this time\\". \\nI will never go back to Flynn\'s b/c of the way this guy treated me and the simple fact that they gave me a used tire!'}
{'label': 4, 'text': "dr. goldberg offers everything i look for in a general practitioner.  he's nice and easy to talk to without being patronizing; he's always on time in seeing his patients; he's affiliated with a top-notch hospita

In [10]:
print("train: ", len(datasets["train"]))
print("validation: ", len(datasets["validation"]))
print("test: ", len(datasets["test"]))

train:  65000
validation:  11700
test:  11700


In [11]:
vectorizer = TfidfVectorizer()
vectorizer.fit_transform(list(datasets["train"]['text']))

<65000x78868 sparse matrix of type '<class 'numpy.float64'>'
	with 5498394 stored elements in Compressed Sparse Row format>

In [38]:
class ZuziaNet(nn.Module):
    def __init__(self, hu=32):
        super().__init__()
        self.fc1 = nn.Linear(78868, hu)
        self.dropout = nn.Dropout(p=0.3)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hu, 5)

    def forward(self, x):
        x = self.fc1(x)
        x = self.dropout(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

In [13]:
def calc_accuracy(logits: torch.Tensor, y_true: torch.Tensor) -> float:
    y_pred_labels = torch.argmax(logits, dim=1).float()
    accuracy = (y_pred_labels == y_true).sum().item() / y_true.size(0)
    return accuracy

In [25]:
def train_network(model: nn.Module, criterion: nn.Module, optimizer: torch.optim.Optimizer,
                  train_loader: DataLoader, val_data: DatasetDict, vectorizer: TfidfVectorizer, n_epochs: int = 200):
    model.train()
    for epoch in range(n_epochs):
        for batch in train_loader:

            x = vectorizer.transform(batch["text"])
            x = torch.tensor(x.toarray(), dtype=torch.float32).to(device)
            
            y = torch.tensor(batch["label"], dtype=torch.long).to(device)
            
            optimizer.zero_grad()
            logits = model(x)
            # logits ma rozmiar (N, 1)
            logits = logits.squeeze(1)
            # logits ma rozmiar (N,)

            # Wyznacz wartość funkcji straty
            loss = criterion(logits, y)

            # Przejście w tył- wyznaczenie gradientu funkcji straty względem parametrów (wag) modelu
            loss.backward()

            # Krok optymalizacji metodą spadku wzdłuż gradientu
            optimizer.step()

        train_accuracy = calc_accuracy(logits, y)

        # Oblicz dokładność klasyfikacji na zbiorze testowym
        
        with torch.inference_mode():
            model.eval()

            x_eval = vectorizer.transform(val_data["text"])
            x_eval = torch.tensor(x_eval.toarray(), dtype=torch.float32, device=device)
            logits = model(x_eval)
            y_eval = torch.tensor(val_data["label"], dtype=torch.long, device=device)
            eval_loss = criterion(logits, y_eval)

            model.train()
            test_accuracy = calc_accuracy(logits.squeeze(1), y_eval)

        wandb.log({"epoch": epoch, "loss": loss.item(), "eval_loss": eval_loss.item()})

        if epoch % 5 == 0:
            print(f"Epoch: {epoch}   Wartość funkcji straty: {loss.item():.5f}   Dokładność (train): {train_accuracy:.4f}   Dokładność (test): {test_accuracy:.4f}")

In [ ]:
net = ZuziaNet()
net.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(net.parameters(), lr=0.001)

train_loader = DataLoader(datasets["train"], batch_size=1024, shuffle=True)

train_network(net, criterion, optimizer, train_loader, datasets["validation"], vectorizer, n_epochs=30)
wandb.finish() 

/net/tscratch/people/plgzwaszczuk/slurm_jobdir/2484643/tmp.t0006/ipykernel_3937585/992692712.py:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y = torch.tensor(batch["label"], dtype=torch.long).to(device)


Epoch: 0   Wartość funkcji straty: 1.48616   Dokładność (train): 0.4672   Dokładność (test): 0.4662
Epoch: 5   Wartość funkcji straty: 0.86125   Dokładność (train): 0.6701   Dokładność (test): 0.5845
Epoch: 10   Wartość funkcji straty: 0.69856   Dokładność (train): 0.7684   Dokładność (test): 0.5816
Epoch: 15   Wartość funkcji straty: 0.54277   Dokładność (train): 0.8074   Dokładność (test): 0.5697
Epoch: 20   Wartość funkcji straty: 0.44892   Dokładność (train): 0.8627   Dokładność (test): 0.5596
Epoch: 25   Wartość funkcji straty: 0.35858   Dokładność (train): 0.8770   Dokładność (test): 0.5504


In [ ]:
def hardest_examples(model, dataset, k=5):
    model.eval()
    with torch.no_grad():
        x = vectorizer.transform(dataset["text"])
        x = torch.tensor(x.toarray(), dtype=torch.float32).to(device)
        logits = model(x)
    
    y_test = list(dataset["label"])
    true_class_logits = logits[range(len(y_test)), y_test]

    hardest_idx = torch.topk(true_class_logits, k, largest=False).indices
    
    print(f"Najtrudniejsze {k} przykłady:")
    for idx in hardest_idx:
        logit = logits[idx]
        true_class = y_test[idx]
        pred_class = logit.argmax().item()
        print(f"Index {idx.item()} | True: {true_class} | Pred: {pred_class} | Logit true class: {logit[true_class].item():.4f}")
        print(f'Text {dataset["text"][idx.item()]}')

In [37]:
hardest_examples(net, datasets["validation"], 10)

Najtrudniejsze 10 przykłady:
Index 6553 | True: 2 | Pred: 0 | Logit true class: -6.8613
Text The pollo especial may be the absolute worst Mexican dish that I have tasted, ever.  The fajitas are the only thing worth eating at this place, everything else I have tried has been extra salty and bland.
Index 3715 | True: 3 | Pred: 0 | Logit true class: -6.5962
Text Great prices, assuming (1) You have the time to dismantle a wrecked or dead car and (2) You know exactly what you are getting into\n\nI got into a major fender bender a few weeks ago. Not wanting to deal with overpriced shipping costs for new parts from the internet (hint: Online stores almost double the amount of money you are paying!), one of my friends told me about this place which I never heard of.\n\nOn my first day and one and a half hour trying to locate a dead 2001 Corolla to salvage for parts, I pulled out a used hood latch that I needed from a 1996 Camry. I already knew off the top of my head that it is going to fit, ba

In [ ]:
wandb.init(
    project="llm_lab1",
    name="mlp_layer_64",
    config={
        "learning_rate": 0.001,
        "batch_size": 32,
        "epochs": 5,
        "hidden_units": 64,
    }
)

net = ZuziaNet(hu=64)
net.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(net.parameters(), lr=0.001)

train_loader = DataLoader(datasets["train"], batch_size=1024, shuffle=True)

train_network(net, criterion, optimizer, train_loader, datasets["validation"], vectorizer, n_epochs=30)

wandb.finish() 

epoch,▁▁▁▁▂▃▃▃▃▄▅▅▅▆▆▇▇▇▇█▁▁▂▂▂▃▃▃▄▄▅▅▅▆▆▇▇▇▇█
eval_loss,█▅▃▂▂▁▁▁▁▁▁▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄
loss,██▇▅▅▄▃▃▃▃▃▃▂▂▂▂▂▂▁▂▁█▆▅▄▄▃▃▃▂▃▂▂▂▂▂▁▁▁▁
epoch,29
eval_loss,1.2105
loss,0.31581


/net/tscratch/people/plgzwaszczuk/slurm_jobdir/2484643/tmp.t0006/ipykernel_3937585/992692712.py:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y = torch.tensor(batch["label"], dtype=torch.long).to(device)


Epoch: 0   Wartość funkcji straty: 1.40311   Dokładność (train): 0.5430   Dokładność (test): 0.5161
Epoch: 5   Wartość funkcji straty: 0.77346   Dokładność (train): 0.6947   Dokładność (test): 0.5884
Epoch: 10   Wartość funkcji straty: 0.57705   Dokładność (train): 0.7828   Dokładność (test): 0.5697
Epoch: 15   Wartość funkcji straty: 0.39152   Dokładność (train): 0.8709   Dokładność (test): 0.5499
Epoch: 20   Wartość funkcji straty: 0.29890   Dokładność (train): 0.9180   Dokładność (test): 0.5362
Epoch: 25   Wartość funkcji straty: 0.22955   Dokładność (train): 0.9344   Dokładność (test): 0.5277


In [40]:
wandb.finish() 

epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
eval_loss,▇▄▂▁▁▁▁▁▁▁▂▂▂▂▃▃▃▄▄▄▅▅▅▆▆▇▇▇██
loss,█▇▅▅▅▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁
epoch,29
eval_loss,1.4642
loss,0.22988


In [ ]:
wandb.init(
    project="llm_lab1",
    name="mlp_layer_64",
    config={
        "learning_rate": 0.001,
        "batch_size": 32,
        "epochs": 5,
        "hidden_units": 64,
    }
)

net = ZuziaNet(hu=64)
net.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(net.parameters(), lr=0.001)

train_loader = DataLoader(datasets["train"], batch_size=1024, shuffle=True)

train_network(net, criterion, optimizer, train_loader, datasets["validation"], vectorizer, n_epochs=30)

wandb.finish() 